In [0]:
"""
id: historical_telephonic_outreach_transcripts
template: source
templateVersion: 1.0.0
name: historical_telephonic_outreach_transcripts
position:
  x: 0
  y: 0
description:
  text: Read data from a specific table.
  hash: bf57b4fd
previewCodeHash: 2687266c33839c62
previewMode: "1000"
config:
  table_source:
    tableName: cmegdemos_catalog.telco_call_center_analytics.historical_telephonic_outreach_transcripts
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "cmegdemos_catalog.telco_call_center_analytics.historical_telephonic_outreach_transcripts"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["historical_telephonic_outreach_transcripts.data"] = out["data"]

In [0]:
"""
id: ai_function_1
template: ai_function
templateVersion: 1.0.0
name: ai_function_1
position:
  x: 300
  y: 0
description:
  text: Add a summary and sentiment analysis of the transcript.
  hash: 6db1d4b7
previewCodeHash: 01ec24308ae0b11d
previewMode: "1000"
config:
  expressions:
    - ai_summarize(transcript, 10) AS `transcript_summary`
    - ai_analyze_sentiment(transcript) AS `sentiment`
input:
  - node: historical_telephonic_outreach_transcripts
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"ai_data": df}

    return {"ai_data": df.selectExpr(*expressions, "*")}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_summarize(transcript, 10) AS `transcript_summary`",
        "ai_analyze_sentiment(transcript) AS `sentiment`"
    ]
}
inputs = {
    "data": ctx["historical_telephonic_outreach_transcripts.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_1.ai_data"] = out["ai_data"]

In [0]:
"""
id: cmegdemos_catalog.default.output_2
template: output
templateVersion: 1.0.0
name: cmegdemos_catalog.network_analytics_enablement.ai_transcript_analysis
position:
  x: 600
  y: 0
description:
  text: Save data to specified table, replacing existing content.
  hash: 718af383
previewCodeHash: 0b50e0152c9a8c76
previewMode: "1000"
config:
  catalog: cmegdemos_catalog
  schema: network_analytics_enablement
  table_name: ai_transcript_analysis
input:
  - node: ai_function_1
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "cmegdemos_catalog",
    "schema": "network_analytics_enablement",
    "table_name": "ai_transcript_analysis"
}
inputs = {
    "data": ctx["ai_function_1.ai_data"]
}
out = run(config, inputs, spark)